<a href="https://colab.research.google.com/github/Thamizhselvan-229/Deep-Learning/blob/main/DL5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader, random_split, Subset
import matplotlib.pyplot as plt
from PIL import Image
import os


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class_names = [
    'airplane', 'automobile', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck'
]


transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        (0.5, 0.5, 0.5),
        (0.5, 0.5, 0.5)
    )
])


dataset = datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transform
)


dataset = Subset(dataset, range(500))

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size]
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False
)


def train_and_evaluate(model, name, num_epochs=5):
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()

    optimizer = optim.Adam(
        model.parameters(),
        lr=1e-4
    )

    train_accs = []
    val_accs = []

    for epoch in range(num_epochs):

        model.train()

        correct = 0
        total = 0

        for inputs, labels in train_loader:

            inputs = inputs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(inputs)

            loss = criterion(outputs, labels)

            loss.backward()

            optimizer.step()

            _, predicted = torch.max(outputs, 1)

            correct += (predicted == labels).sum().item()
            total += labels.size(0)

        train_accuracy = 100 * correct / total
        train_accs.append(train_accuracy)

        model.eval()

        correct = 0
        total = 0

        with torch.no_grad():

            for inputs, labels in val_loader:

                inputs = inputs.to(device)
                labels = labels.to(device)

                outputs = model(inputs)

                _, predicted = torch.max(outputs, 1)

                correct += (predicted == labels).sum().item()
                total += labels.size(0)

        val_accuracy = 100 * correct / total
        val_accs.append(val_accuracy)

        print(
            f"{name} Epoch {epoch + 1}/{num_epochs} - "
            f"Train Acc: {train_accuracy:.2f}%, "
            f"Val Acc: {val_accuracy:.2f}%"
        )

    return model, train_accs, val_accs


def get_model(name):

    if name == "vgg":
        model = models.vgg16(pretrained=True)
        model.classifier[6] = nn.Linear(4096, 10)

    elif name == "resnet":
        model = models.resnet18(pretrained=True)
        model.fc = nn.Linear(
            model.fc.in_features,
            10
        )

    elif name == "googlenet":
        model = models.googlenet(
            pretrained=True,
            aux_logits=True
        )

        model.fc = nn.Linear(
            model.fc.in_features,
            10
        )

    else:
        raise ValueError("Unknown model")

    return model


results = {}
trained_models = {}

for model_name in ["vgg", "resnet", "googlenet"]:

    model = get_model(model_name)

    trained_model, train_acc, val_acc = train_and_evaluate(
        model,
        model_name.upper(),
        num_epochs=5
    )

    results[model_name] = (
        train_acc,
        val_acc
    )

    trained_models[model_name] = trained_model


plt.figure(figsize=(10, 6))

for name, (train_acc, val_acc) in results.items():

    plt.plot(
        val_acc,
        label=f'{name.upper()} Val Acc'
    )

plt.title(
    'Validation Accuracy Comparison on CIFAR-10'
)

plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')

plt.legend()
plt.grid(True)

plt.show()


def predict_image(image_path, models_dict):

    image = Image.open(
        image_path
    ).convert('RGB')

    image = transform(image)

    image = image.unsqueeze(0)

    image = image.to(device)

    print(
        f"\nPrediction results for image: {image_path}"
    )

    for model_name, model in models_dict.items():

        model.eval()

        with torch.no_grad():

            outputs = model(image)

            _, predicted = outputs.max(1)

            pred_class = class_names[
                predicted.item()
            ]

        print(
            f"{model_name.upper():<10} => {pred_class}"
        )


custom_image_path = "download.jpeg"

if os.path.exists(custom_image_path):

    predict_image(
        custom_image_path,
        trained_models
    )

else:

    print(
        f"\nImage not found: {custom_image_path}"
    )


100%|██████████| 170M/170M [01:16<00:00, 2.24MB/s]
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:05<00:00, 98.8MB/s]


VGG Epoch 1/5 - Train Acc: 26.25%, Val Acc: 44.00%
VGG Epoch 2/5 - Train Acc: 65.00%, Val Acc: 61.00%
